# Multilayer Perceptron (Neural Network) Model for Laptop Price Prediction

This notebook trains a Multilayer Perceptron with hyperparameter tuning, evaluates performance metrics, and compares with other models.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pydantic import BaseModel, Field
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Load preprocessed data
train_df_encoded = pd.read_csv('../../data/processed/training/train_data_v0.csv')
val_df_encoded = pd.read_csv('../../data/processed/training/val_data_v0.csv')
test_df_encoded = pd.read_csv('../../data/processed/training/test_data_v0.csv')

print(f"Train set: {train_df_encoded.shape}")
print(f"Validation set: {val_df_encoded.shape}")
print(f"Test set: {test_df_encoded.shape}")

In [ ]:
# ==================== DATA PREPARATION ====================
X_train = train_df_encoded.drop('Price', axis=1).values
y_train = train_df_encoded['Price'].values
X_val = val_df_encoded.drop('Price', axis=1).values
y_val = val_df_encoded['Price'].values
X_test = test_df_encoded.drop('Price', axis=1).values
y_test = test_df_encoded['Price'].values

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# ==================== STANDARDIZE FEATURES (REQUIRED FOR MLP) ====================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nScaler fitted and applied to all sets")

# ==================== TRAIN MULTILAYER PERCEPTRON MODEL ====================
# Grid search for best hyperparameters
print("\n" + "="*80)
print("GRID SEARCH: Finding optimal MLP hyperparameters")
print("="*80)

best_val_mse = float('inf')
best_params = {}
best_model = None

# Grid search parameters
hidden_layers = [(128, 64), (256, 128, 64), (64, 32), (100, 50, 25)]
learning_rates = [0.001, 0.01, 0.0001]

for hidden_layer in hidden_layers:
    for lr in learning_rates:
        mlp = MLPRegressor(
            hidden_layer_sizes=hidden_layer,
            learning_rate_init=lr,
            max_iter=1000,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20
        )
        mlp.fit(X_train_scaled, y_train)
        
        y_val_pred = mlp.predict(X_val_scaled)
        val_mse = mean_squared_error(y_val, y_val_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        val_mape = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
        
        layer_str = f"{hidden_layer}".replace(', ', '-')
        print(f"Layers={layer_str:20s}, LR={lr:7.4f}: MSE={val_mse:12,.0f} | MAE=${val_mae:8.2f} | MAPE={val_mape:6.2f}%")
        
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_params = {'hidden_layer_sizes': hidden_layer, 'learning_rate_init': lr}
            best_model = mlp

print(f"\n{'▼'*80}")
print(f"BEST PARAMS: hidden_layers={best_params['hidden_layer_sizes']}, lr={best_params['learning_rate_init']}")
print(f"Best Validation MSE: {best_val_mse:,.2f}")
print(f"{'▼'*80}")

# Use best model
mlp_model = best_model

# ==================== EVALUATE MODEL ====================
print("\n" + "="*80)
print("MODEL EVALUATION ON ALL DATASETS")
print("="*80)

# Training set evaluation
y_train_pred = mlp_model.predict(X_train_scaled)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mape = np.mean(np.abs((y_train - y_train_pred) / y_train)) * 100
train_r2 = r2_score(y_train, y_train_pred)

print(f"\n{'TRAINING SET':-^80}")
print(f"  • MSE:    {train_mse:>15,.2f}")
print(f"  • RMSE:   ${train_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${train_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {train_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {train_r2:>15.4f}  (Explains {train_r2*100:.1f}% of price variance)")

# Validation set evaluation
y_val_pred = mlp_model.predict(X_val_scaled)
val_mse = mean_squared_error(y_val, y_val_pred)
val_rmse = np.sqrt(val_mse)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_mape = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
val_r2 = r2_score(y_val, y_val_pred)

print(f"\n{'VALIDATION SET':-^80}")
print(f"  • MSE:    {val_mse:>15,.2f}")
print(f"  • RMSE:   ${val_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${val_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {val_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {val_r2:>15.4f}  (Explains {val_r2*100:.1f}% of price variance)")

# Test set evaluation
y_test_pred = mlp_model.predict(X_test_scaled)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n{'TEST SET':-^80}")
print(f"  • MSE:    {test_mse:>15,.2f}")
print(f"  • RMSE:   ${test_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${test_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {test_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {test_r2:>15.4f}  (Explains {test_r2*100:.1f}% of price variance)")

print("\n✓ Model training complete!")

# Save model
joblib.dump(mlp_model, 'mlp_model.pkl')
joblib.dump(scaler, 'mlp_scaler.pkl')
print("✓ Model and scaler saved")

# Store feature names
feature_names = list(train_df_encoded.drop('Price', axis=1).columns)
print(f"\nFeature names ({len(feature_names)} total): {feature_names}")

In [ ]:
# ==================== INFERENCE FUNCTIONS ====================

def infer_price_from_encoded_data_mlp(encoded_data: pd.DataFrame) -> float:
    """Predict laptop price from already encoded data using MLP"""
    X = encoded_data.values
    X_scaled = scaler.transform(X)
    price = mlp_model.predict(X_scaled)[0]
    return price

def infer_price_from_raw_input_mlp(laptop_spec) -> dict:
    """Predict laptop price from raw human input using MLP"""
    # Get CPU and GPU marks (placeholder - to be implemented)
    cpu_mark = 12675.0
    gpu_mark = 2000.0
    
    # Parse resolution
    width, height = map(int, laptop_spec.resolution.split('x'))
    
    # Parse RAM and Storage
    if laptop_spec.ram.upper().endswith('TB'):
        ram_gb = float(laptop_spec.ram[:-2]) * 1024
    else:
        ram_gb = float(laptop_spec.ram.replace('GB', '').strip())
    
    if laptop_spec.storage.upper().endswith('TB'):
        storage_gb = float(laptop_spec.storage[:-2]) * 1024
    else:
        storage_gb = float(laptop_spec.storage.replace('GB', '').strip())
    
    # Create feature dictionary
    data_dict = {}
    data_dict['CPU Mark'] = cpu_mark
    data_dict['GPU Mark'] = gpu_mark
    data_dict['Monitor'] = float(laptop_spec.monitor)
    data_dict['Width'] = width
    data_dict['Height'] = height
    data_dict['RAM'] = ram_gb
    data_dict['Storage Amount'] = storage_gb
    data_dict['Weight'] = float(laptop_spec.weight)
    
    # Initialize all categorical features to 0
    known_brands = ['Brand_Acer', 'Brand_Apple', 'Brand_Asus', 'Brand_Dell', 'Brand_HP', 
                    'Brand_LG', 'Brand_Lenovo', 'Brand_MSI', 'Brand_Microsoft', 'Brand_Other']
    known_os = ['OS_Other', 'OS_Windows 10', 'OS_Windows 11']
    
    for brand_col in known_brands:
        data_dict[brand_col] = 0
    for os_col in known_os:
        data_dict[os_col] = 0
    
    # Set brand
    brand_clean = laptop_spec.brand.lower()
    brand_mapping = {
        'acer': 'Brand_Acer', 'apple': 'Brand_Apple', 'asus': 'Brand_Asus',
        'dell': 'Brand_Dell', 'hp': 'Brand_HP', 'lg': 'Brand_LG',
        'lenovo': 'Brand_Lenovo', 'msi': 'Brand_MSI', 'microsoft': 'Brand_Microsoft'
    }
    
    if brand_clean in brand_mapping:
        data_dict[brand_mapping[brand_clean]] = 1
    else:
        data_dict['Brand_Other'] = 1
    
    # Set OS
    os_clean = laptop_spec.os.lower()
    if 'windows 11' in os_clean:
        data_dict['OS_Windows 11'] = 1
    elif 'windows 10' in os_clean:
        data_dict['OS_Windows 10'] = 1
    else:
        data_dict['OS_Other'] = 1
    
    # Create DataFrame
    X = pd.DataFrame([data_dict])[feature_names]
    X_scaled = scaler.transform(X.values)
    predicted_price = mlp_model.predict(X_scaled)[0]
    
    return {
        'predicted_price': float(predicted_price),
        'specification': laptop_spec.model_dump(),
        'cpu_mark': cpu_mark,
        'gpu_mark': gpu_mark
    }


# ==================== TEST INFERENCE ====================
print("\n" + "="*80)
print("INFERENCE TEST: Using encoded data from test set")
print("="*80)
sample_encoded = test_df_encoded.drop('Price', axis=1).iloc[0:1]
actual_price = test_df_encoded['Price'].iloc[0]
predicted_price_mlp = infer_price_from_encoded_data_mlp(sample_encoded)
error = abs(predicted_price_mlp - actual_price)
error_pct = (error / actual_price) * 100

print(f"Actual Price:    ${actual_price:.2f}")
print(f"Predicted Price: ${predicted_price_mlp:.2f}")
print(f"Error:           ${error:.2f} ({error_pct:.1f}%)")
print("="*80)